In [2]:
import pandas as pd

df = pd.read_csv("output/raw_csv/output_constituency.csv")

# ดูค่า ตำบล ที่ผิดปกติ
print(df["ตำบล"].value_counts().to_string())


ตำบล
ท่าเรือ                251
ท้ายสำเภา.pdf          201
คลองน้อย               170
ช้างซ้าย.pdf           151
กำแพงเซา               150
นาสาร.pdf              141
คลองกระบือ             130
นาพรุ.pdf              110
ท่าไร่                 100
ไชยมนตรี                90
ปากพนังฝั่งตะวันออก     90
ป่าระกำ                 87
มะม่วงสองต้น            82
เกาะทวด                 80
บ้านใหม่                70
ชะเมา                   70
ปากพนังฝั่งตะวันตก      60
แหลมตะลุมพุก            40
อำเภอ                   20
ปากพนัง                 20
บางจาก_8.pdf            11
เมืองปากเกร็ดอำเภอ      10
มากเพียง                10
เมืองปากพนัง            10
บางพระ                  10
บางจาก_12.pdf           10
บางโตรุย                10
ปากคลอง                 10
เลื่องปากพนัง           10
ปากน้ำเริง              10
เมืองปากเกร็ด           10
จังหวัด                 10
ปากน้ำ                  10
เมืองปากพนเข้าอำเภอ     10
เสนาะแพรกษา             10
เสียหายหน้า             10
เลือกตั้งฯ             

In [3]:
import re

def clean_tambon(val):
    if pd.isna(val):
        return val
    
    # ตัด .pdf และ _X.pdf ออก
    val = re.sub(r'_?\d*\.pdf.*$', '', val, flags=re.IGNORECASE)
    
    # strip whitespace
    val = val.strip()
    
    return val

df["ตำบล"] = df["ตำบล"].apply(clean_tambon)

# ดูผลที่ยังผิดอยู่
print(df["ตำบล"].value_counts().to_string())


ตำบล
ท่าเรือ                251
ท้ายสำเภา              201
คลองน้อย               170
ช้างซ้าย               151
กำแพงเซา               150
นาสาร                  141
คลองกระบือ             130
บางจาก                 121
นาพรุ                  110
ท่าไร่                 100
ปากพนังฝั่งตะวันออก     90
ไชยมนตรี                90
ป่าระกำ                 87
มะม่วงสองต้น            82
เกาะทวด                 80
ชะเมา                   70
บ้านใหม่                70
ปากพนังฝั่งตะวันตก      60
แหลมตะลุมพุก            40
ปากพนัง                 20
อำเภอ                   20
บางโตรุย                10
ปากน้ำ                  10
จังหวัด                 10
เมืองปากเกร็ด           10
ปากน้ำเริง              10
เลื่องปากพนัง           10
ปากคลอง                 10
เมืองปากพนเข้าอำเภอ     10
มากเพียง                10
บางพระ                  10
เมืองปากพนัง            10
เมืองปากเกร็ดอำเภอ      10
เสียหายหน้า             10
เลือกตั้งฯ              10
เสนาะแพรกษา             10
บางจาก_1.              

In [25]:
tambon_map = {
    "อำเภอ":           None,        # OCR อ่านผิดทั้งหมด → null ไว้ก่อน
    "เมืองปากเกร็ดอำเภอ": None,
    "มากเพียง":        None,
    "เมืองปากพนัง":    "ปากพนัง",   # เทศบาลเมืองปากพนัง
    "ท่าศาลา":         None,        # ไม่แน่ใจ → null ไว้ก่อน
    "บางจาก_1.": "บางจาก",
    "บางจาก_2.": "บางจาก",
    "เมืองปากพนเข้าอำเภอ": None,
    "มากเพียง": None,
    "เมืองปากพนัง": None,
    "เสียหายหน้า": None,
    "เลือกตั้งฯ": None,
    "เสนาะแพรกษา": None,
    "เมืองปากเกร็ด": None,
    "เลื่องปากพนัง": None,
    "จังหวัด": None,
    "ปากน้ำเริง": None,
    "ปากน้ำ": None,
    "ปากพนัง": None,
    "บางพระ": None,
    "บางโตรุย": None,
    "ปากคลอง": None,
    "เทศบาลเมืองปากพนัง": "เมืองปากพนัง"
}

In [26]:
df["ตำบล"] = df["ตำบล"].replace(tambon_map)

print(df["ตำบล"].value_counts().to_string())

ตำบล
เมืองปากพนัง           268
ท่าเรือ                251
ท้ายสำเภา              201
คลองน้อย               170
ช้างซ้าย               151
กำแพงเซา               150
บางจาก                 141
นาสาร                  141
คลองกระบือ             130
นาพรุ                  110
ท่าไร่                 100
ไชยมนตรี                90
ปากพนังฝั่งตะวันออก     90
ป่าระกำ                 87
มะม่วงสองต้น            82
เกาะทวด                 80
ชะเมา                   70
บ้านใหม่                70
ปากพนังฝั่งตะวันตก      60
แหลมตะลุมพุก            40


In [24]:
def extract_tambon_from_path(source_file):
    """ดึงชื่อตำบลจาก path เช่น data/raw_jpg/ตำบลท่าเรือ/... → ท่าเรือ"""
    if pd.isna(source_file):
        return None
    # หา folder ที่ขึ้นต้นด้วย ตำบล หรือ เทศบาล
    parts = str(source_file).replace("\\", "/").split("/")
    for part in parts:
        if part.startswith("ตำบล"):
            return part.replace("ตำบล", "").strip()
        if part.startswith("เทศบาล"):
            return part.strip()
    return None

# Fill null ตำบล จาก path
df["ตำบล"] = df.apply(
    lambda row: extract_tambon_from_path(row["source_file"])
    if pd.isna(row["ตำบล"]) else row["ตำบล"],
    axis=1
)

print("null remaining:", df["ตำบล"].isna().sum())
print(df["ตำบล"].value_counts().to_string())

null remaining: 0
ตำบล
เทศบาลเมืองปากพนัง     268
ท่าเรือ                251
ท้ายสำเภา              201
คลองน้อย               170
ช้างซ้าย               151
กำแพงเซา               150
บางจาก                 141
นาสาร                  141
คลองกระบือ             130
นาพรุ                  110
ท่าไร่                 100
ไชยมนตรี                90
ปากพนังฝั่งตะวันออก     90
ป่าระกำ                 87
มะม่วงสองต้น            82
เกาะทวด                 80
ชะเมา                   70
บ้านใหม่                70
ปากพนังฝั่งตะวันตก      60
แหลมตะลุมพุก            40


In [19]:
# ดู source_file pattern ก่อน
print(df[df["อำเภอ"].isna()]["source_file"].unique()[:10])

['data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 13/5.18/1770580262049-aae396df-27e3-42b0-81fd-5027d3422757.jpg'
 'data/raw_jpg/ตำบลท่าเรือ/หน่วยเลือกตั้งที่ 10/5.18/1770696981084-f28e8847-c06f-4c42-b50a-45d27ce40c7d.jpg'
 'data/raw_jpg/ตำบลท่าเรือ/หน่วยเลือกตั้งที่ 24/5.18/1770695470603-ce43ac28-608f-4bdd-8873-509a25a9a9de.jpg'
 'data/raw_pdf/ตำบลคลองน้อย/18/เขต 2 ตำบลคลองน้อย.pdf#p5'
 'data/raw_pdf/ตำบลคลองน้อย/18/เขต 2 ตำบลคลองน้อย.pdf#p14'
 'data/raw_pdf/ตำบลชะเมา/18/เขต 2 ต.ชะเมา 5.18.pdf#p3'
 'data/raw_pdf/ตำบลช้างซ้าย/18/ตำบลช้างซ้าย.pdf#p7'
 'data/raw_pdf/ตำบลท้ายสำเภา/18/ตำบลท้ายสำเภา.pdf#p20'
 'data/raw_pdf/ตำบลนาพรุ/18/ตำบลนาพรุ.pdf'
 'data/raw_pdf/ตำบลนาพรุ/18/ตำบลนาพรุ.pdf#p2']


In [20]:
def fill_amphoe(row):
    if pd.notna(row["อำเภอ"]):
        return row["อำเภอ"]
    source = str(row["source_file"])
    if "raw_jpg" in source:
        return "เมือง"
    if "raw_pdf" in source:
        return "ปากพนัง"
    return None

df["อำเภอ"] = df.apply(fill_amphoe, axis=1)

print(df["อำเภอ"].value_counts())
print("null remaining:", df["อำเภอ"].isna().sum())

อำเภอ
เมือง        471
ปากพนัง      471
พระนคร       190
ปากเกร็ด     140
จังหวัด       69
            ... 
ปากคลัง       10
ตำบล          10
ปัตตานี       10
ป              5
ฝ่ายหนึ่ง      1
Name: count, Length: 90, dtype: int64
null remaining: 0


In [21]:
# ดูว่า null คะแนน มาจากผู้สมัครหมายเลขไหนบ้าง
print(df[df["คะแนน"].isna()][["ตำบล", "หน่วยเลือกตั้งที่", "หมายเลข", "ชื่อสกุล", "source_file"]].to_string())


                     ตำบล  หน่วยเลือกตั้งที่  หมายเลข                 ชื่อสกุล                                                                                                source_file
86               กำแพงเซา                3.0      7.0     นายปัญโน หวานนุรักษ์  data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 3/5.18/1770579270147-02760d4a-ea74-49dc-be79-47ec7b17aeb9.jpg
99               กำแพงเซา                4.0     10.0       นายนพดล กาญจนรักษ์  data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 4/5.18/1770576843377-033e2c1c-8996-4756-8964-328d1059ce59.jpg
230               ท่าเรือ               17.0      8.0          นายทศพร แซ่ซิ้น  data/raw_jpg/ตำบลท่าเรือ/หน่วยเลือกตั้งที่ 17/5.18/1770697884731-e60216b6-ea82-49d0-84ec-912880952af4.jpg
517              ไชยมนตรี                3.0      7.0     นายปัญโน หวานนุรักษ์  data/raw_jpg/ตำบลไชยมนตรี/หน่วยเลือกตั้งที่ 3/5.18/1770578186945-63612c86-e4a3-412b-af8b-453f68607947.jpg
519              ไชยมนตรี                3.0      9.0      นายสิทธิพงศ

In [27]:
# Mapping ตำบล → อำเภอ
tambon_to_amphoe = {
    # อำเภอพระพรหม
    "นาสาร":                  "พระพรหม",
    "นาพรุ":                   "พระพรหม",
    "ท้ายสำเภา":               "พระพรหม",
    "ช้างซ้าย":                "พระพรหม",

    # อำเภอเมือง
    "ไชยมนตรี":                "เมือง",
    "บางจาก":                  "เมือง",
    "ท่าไร่":                  "เมือง",
    "มะม่วงสองต้น":            "เมือง",
    "ท่าเรือ":                 "เมือง",
    "กำแพงเซา":                "เมือง",

    # อำเภอปากพนัง
    "แหลมตะลุมพุก":            "ปากพนัง",
    "เกาะทวด":                 "ปากพนัง",
    "ปากพนังฝั่งตะวันออก":     "ปากพนัง",
    "ป่าระกำ":                 "ปากพนัง",
    "เมืองปากพนัง":                 "ปากพนัง",  # เทศบาลเมืองปากพนัง
    "ปากพนังฝั่งตะวันตก":     "ปากพนัง",
    "บ้านใหม่":                "ปากพนัง",
    "ชะเมา":                   "ปากพนัง",
    "คลองกระบือ":              "ปากพนัง",
    "คลองน้อย":                "ปากพนัง",
}

# Fill อำเภอ จาก mapping
df["อำเภอ"] = df.apply(
    lambda row: tambon_to_amphoe.get(row["ตำบล"], row["อำเภอ"])
    if pd.isna(row["อำเภอ"]) else row["อำเภอ"],
    axis=1
)

print(df["อำเภอ"].value_counts())
print("null remaining:", df["อำเภอ"].isna().sum())


อำเภอ
เมือง        471
ปากพนัง      471
พระนคร       190
ปากเกร็ด     140
จังหวัด       69
            ... 
ปากคลัง       10
ตำบล          10
ปัตตานี       10
ป              5
ฝ่ายหนึ่ง      1
Name: count, Length: 90, dtype: int64
null remaining: 0


In [28]:
# ดูว่า อำเภอแปลกๆ มาจากตำบลไหน
print(df[~df["อำเภอ"].isin(["เมือง", "ปากพนัง", "พระพรหม"])][["ตำบล", "อำเภอ", "source_file"]].drop_duplicates().to_string())


                     ตำบล                                           อำเภอ                                                                                                source_file
70               กำแพงเซา                                       กำแพงเพชร                                                data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 2/5.18/สส-1.jpg
80               กำแพงเซา                                         เชิงชุม  data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 3/5.18/1770579270147-02760d4a-ea74-49dc-be79-47ec7b17aeb9.jpg
90               กำแพงเซา                                         เข็มทอง  data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 4/5.18/1770576843377-033e2c1c-8996-4756-8964-328d1059ce59.jpg
100              กำแพงเซา                                            น้อย                                              data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 5/5.18/สส-1_0.jpg
110              กำแพงเซา                                              มี                      

In [29]:
# Override อำเภอ ทั้งหมดจาก tambon_to_amphoe (ไม่ใช่แค่ null)
df["อำเภอ"] = df["ตำบล"].map(tambon_to_amphoe)

print(df["อำเภอ"].value_counts())
print("null remaining:", df["อำเภอ"].isna().sum())


อำเภอ
ปากพนัง    1065
เมือง       814
พระพรหม     603
Name: count, dtype: int64
null remaining: 0


In [30]:
ok_df = df[df["needs_review"] == "NO"]

print(f"Total NO: {len(ok_df)} rows, {ok_df['source_file'].nunique()} หน่วย")
print()

# ดูทีละหน่วย — คะแนนรวมแต่ละหน่วย
summary = (
    ok_df.groupby(["ตำบล", "หน่วยเลือกตั้งที่", "source_file"])
    .agg(
        รวมคะแนน=("คะแนน", "sum"),
        บัตรดี=("จำนวนบัตรดี", "first"),
        ผู้สมัคร=("หมายเลข", "count"),
    )
    .reset_index()
)

print(summary.to_string())


Total NO: 231 rows, 24 หน่วย

            ตำบล  หน่วยเลือกตั้งที่                                                             source_file  รวมคะแนน  บัตรดี  ผู้สมัคร
0       กำแพงเซา                5.0           data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 5/5.18/สส-1_0.jpg     545.0     NaN        10
1       คลองน้อย                6.0                  data/raw_pdf/ตำบลคลองน้อย/18/เขต 2 ตำบลคลองน้อย.pdf#p3     124.0     NaN        10
2       คลองน้อย               10.0                 data/raw_pdf/ตำบลคลองน้อย/18/เขต 2 ตำบลคลองน้อย.pdf#p10     473.0     NaN        10
3       คลองน้อย               18.0                 data/raw_pdf/ตำบลคลองน้อย/18/เขต 2 ตำบลคลองน้อย.pdf#p18     361.0     NaN        10
4       ช้างซ้าย                9.0                        data/raw_pdf/ตำบลช้างซ้าย/18/ตำบลช้างซ้าย.pdf#p7     183.0     NaN        10
5       ช้างซ้าย               16.0                       data/raw_pdf/ตำบลช้างซ้าย/18/ตำบลช้างซ้าย.pdf#p16     423.0     NaN        10
6         ท่าไร่  

In [31]:
df.columns

Index(['จังหวัด', 'เขตเลือกตั้งที่', 'ตำบล', 'อำเภอ', 'หน่วยเลือกตั้งที่',
       'หมู่ที่', 'จำนวนผู้มีสิทธิเลือกตั้ง', 'จำนวนผู้มาแสดงตน',
       'จำนวนบัตรที่ได้รับจัดสรร', 'จำนวนบัตรที่ใช้', 'จำนวนบัตรดี',
       'จำนวนบัตรเสีย', 'จำนวนบัตรที่ไม่เลือกผู้สมัคร', 'จำนวนบัตรคงเหลือ',
       'source_file', 'needs_review', 'validation_notes', 'หมายเลข',
       'ชื่อสกุล', 'พรรค', 'คะแนน', 'match_method', 'match_score'],
      dtype='object')

In [32]:
# Export ทั้งหมดออกไปแก้ใน Excel
df.to_csv("output/raw_csv/output_constituency_to_fix.csv", index=False, encoding="utf-8-sig")
print(f"✅ Export {len(df)} rows → output/raw_csv/output_constituency_to_fix.csv")


✅ Export 2482 rows → output/raw_csv/output_constituency_to_fix.csv


In [33]:
import pandas as pd

df = pd.read_csv("output/cleaned/constituency_clean.csv")

# Fill คะแนน null → 0
df["คะแนน"] = df["คะแนน"].fillna(0).astype(int)

print(f"Shape: {df.shape}")
print(f"null คะแนน remaining: {df['คะแนน'].isna().sum()}")
print()

# Re-validate logic ทั้ง 3 checks
results = []
for (tambon, unit, source), group in df.groupby(["ตำบล", "หน่วยเลือกตั้งที่", "source_file"]):
    row = group.iloc[0]
    total_score = group["คะแนน"].sum()
    issues = []

    # Check 1
    alloc = row["จำนวนบัตรที่ได้รับจัดสรร"]
    used  = row["จำนวนบัตรที่ใช้"]
    rem   = row["จำนวนบัตรคงเหลือ"]
    if pd.notna(alloc) and pd.notna(used) and pd.notna(rem):
        if rem != alloc - used:
            issues.append(f"check1: {rem} ≠ {alloc}-{used}={alloc-used}")

    # Check 2
    good    = row["จำนวนบัตรดี"]
    spoiled = row["จำนวนบัตรเสีย"]
    novote  = row["จำนวนบัตรที่ไม่เลือกผู้สมัคร"]
    if pd.notna(good) and pd.notna(spoiled) and pd.notna(novote) and pd.notna(used):
        if good + spoiled + novote != used:
            issues.append(f"check2: {good}+{spoiled}+{novote}={good+spoiled+novote} ≠ {used}")

    # Check 3
    if pd.notna(good):
        if total_score != good:
            issues.append(f"check3: รวมคะแนน={total_score} ≠ บัตรดี={good}")

    results.append({
        "ตำบล": tambon,
        "หน่วยเลือกตั้งที่": unit,
        "pass": len(issues) == 0,
        "issues": " | ".join(issues),
    })

result_df = pd.DataFrame(results)
print(f"✅ Pass: {result_df['pass'].sum()} หน่วย")
print(f"❌ Fail: {(~result_df['pass']).sum()} หน่วย")
print()
print("ตัวอย่างที่ยัง fail:")
print(result_df[~result_df["pass"]].head(10).to_string())


Shape: (2560, 23)
null คะแนน remaining: 0

✅ Pass: 200 หน่วย
❌ Fail: 77 หน่วย

ตัวอย่างที่ยัง fail:
          ตำบล  หน่วยเลือกตั้งที่   pass                             issues
1     กำแพงเซา                  2  False  check3: รวมคะแนน=369 ≠ บัตรดี=363
2     กำแพงเซา                  3  False  check3: รวมคะแนน=391 ≠ บัตรดี=313
6     กำแพงเซา                  7  False  check3: รวมคะแนน=321 ≠ บัตรดี=301
10    กำแพงเซา                 11  False          check1: 137 ≠ 500-362=138
18  คลองกระบือ                  4  False  check3: รวมคะแนน=172 ≠ บัตรดี=139
19  คลองกระบือ                  5  False  check3: รวมคะแนน=224 ≠ บัตรดี=195
26  คลองกระบือ                 12  False  check3: รวมคะแนน=346 ≠ บัตรดี=304
27  คลองกระบือ                 13  False  check3: รวมคะแนน=258 ≠ บัตรดี=268
31    คลองน้อย                  4  False  check3: รวมคะแนน=309 ≠ บัตรดี=306
33    คลองน้อย                  5  False    check3: รวมคะแนน=0 ≠ บัตรดี=260


In [35]:
df = df.drop(columns=["needs_review", "validation_notes","match_method", "match_score"], errors="ignore")
print(df.columns.tolist())


['จังหวัด', 'เขตเลือกตั้งที่', 'ตำบล', 'อำเภอ', 'หน่วยเลือกตั้งที่', 'หมู่ที่', 'จำนวนผู้มีสิทธิเลือกตั้ง', 'จำนวนผู้มาแสดงตน', 'จำนวนบัตรที่ได้รับจัดสรร', 'จำนวนบัตรที่ใช้', 'จำนวนบัตรดี', 'จำนวนบัตรเสีย', 'จำนวนบัตรที่ไม่เลือกผู้สมัคร', 'จำนวนบัตรคงเหลือ', 'source_file', 'หมายเลข', 'ชื่อสกุล', 'พรรค', 'คะแนน']


In [36]:
# รวมคะแนนต่อหน่วย แล้วเทียบกับบัตรดี
check = (
    df.groupby(["ตำบล", "หน่วยเลือกตั้งที่", "source_file"])
    .agg(
        รวมคะแนน=("คะแนน", "sum"),
        บัตรดี=("จำนวนบัตรดี", "first"),
    )
    .reset_index()
)

check["ผ่าน"] = check["รวมคะแนน"] == check["บัตรดี"]
check["ต่าง"] = check["รวมคะแนน"] - check["บัตรดี"]

print(f"✅ ผ่าน: {check['ผ่าน'].sum()} หน่วย")
print(f"❌ ไม่ผ่าน: {(~check['ผ่าน']).sum()} หน่วย")
print()
print("หน่วยที่ไม่ผ่าน:")
print(check[~check["ผ่าน"]][["ตำบล", "หน่วยเลือกตั้งที่", "รวมคะแนน", "บัตรดี", "ต่าง", "source_file"]].to_string())


✅ ผ่าน: 207 หน่วย
❌ ไม่ผ่าน: 70 หน่วย

หน่วยที่ไม่ผ่าน:
                    ตำบล  หน่วยเลือกตั้งที่  รวมคะแนน  บัตรดี  ต่าง                                                                                                source_file
1               กำแพงเซา                  2       369     363     6                                                data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 2/5.18/สส-1.jpg
2               กำแพงเซา                  3       391     313    78  data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 3/5.18/1770579270147-02760d4a-ea74-49dc-be79-47ec7b17aeb9.jpg
6               กำแพงเซา                  7       321     301    20                                              data/raw_jpg/ตำบลกำแพงเซา/หน่วยเลือกตั้งที่ 7/5.18/สส-1_0.jpg
18            คลองกระบือ                  4       172     139    33                                         data/raw_pdf/ตำบลคลองกระบือ/18/เขต 2 ต.คลองกระบือ อ.ปากพนัง.pdf#p4
19            คลองกระบือ                  5       224     195    29  

In [37]:
# Export หน่วยที่ไม่ผ่าน check3
check[~check["ผ่าน"]].to_csv(
    "output/cleaned/failed_check3.csv", 
    index=False, encoding="utf-8-sig"
)
print(f"✅ Export failed_check3.csv — {(~check['ผ่าน']).sum()} หน่วย")

# Export cleaned data
df.to_csv(
    "output/cleaned/constituency_clean.csv", 
    index=False, encoding="utf-8-sig"
)
print(f"✅ Export constituency_clean.csv — {len(df)} rows")


✅ Export failed_check3.csv — 70 หน่วย
✅ Export constituency_clean.csv — 2560 rows


In [38]:
df = pd.read_csv("output/cleaned/constituency_cleaned.csv")

# เอา source_file ออก
df = df.drop(columns=["source_file"], errors="ignore")

print(df.shape)
print(df.columns.tolist())
df.head()


(2560, 18)
['จังหวัด', 'เขตเลือกตั้งที่', 'ตำบล', 'อำเภอ', 'หน่วยเลือกตั้งที่', 'หมู่ที่', 'จำนวนผู้มีสิทธิเลือกตั้ง', 'จำนวนผู้มาแสดงตน', 'จำนวนบัตรที่ได้รับจัดสรร', 'จำนวนบัตรที่ใช้', 'จำนวนบัตรดี', 'จำนวนบัตรเสีย', 'จำนวนบัตรที่ไม่เลือกผู้สมัคร', 'จำนวนบัตรคงเหลือ', 'หมายเลข', 'ชื่อสกุล', 'พรรค', 'คะแนน']


,จังหวัด,เขตเลือกตั้งที่,ตำบล,อำเภอ,หน่วยเลือกตั้งที่,หมู่ที่,จำนวนผู้มีสิทธิเลือกตั้ง,จำนวนผู้มาแสดงตน,จำนวนบัตรที่ได้รับจัดสรร,จำนวนบัตรที่ใช้,จำนวนบัตรดี,จำนวนบัตรเสีย,จำนวนบัตรที่ไม่เลือกผู้สมัคร,จำนวนบัตรคงเหลือ,หมายเลข,ชื่อสกุล,พรรค,คะแนน
0,นครศรีธรรมราช,2,กำแพงเซา,เมือง,1,1.0,547,410.0,540,410,356,13,41,130,1,นางสาวนันทวัน วิเชียร,ภูมิใจไทย,186
1,นครศรีธรรมราช,2,กำแพงเซา,เมือง,1,1.0,547,410.0,540,410,356,13,41,130,2,นายนนทิวรรธน์ นนทภักดิ์,รวมไทยสร้างชาติ,11
2,นครศรีธรรมราช,2,กำแพงเซา,เมือง,1,1.0,547,410.0,540,410,356,13,41,130,3,นายเชาวน์วัศ เสนพงศ์,ประชาธิปัตย์,79
3,นครศรีธรรมราช,2,กำแพงเซา,เมือง,1,1.0,547,410.0,540,410,356,13,41,130,4,นายเจริญ ชินวงศ์,กล้าธรรม,9
4,นครศรีธรรมราช,2,กำแพงเซา,เมือง,1,1.0,547,410.0,540,410,356,13,41,130,5,นายจรยุทธ มาศบำรุง,ประชาชน,61


In [39]:
import os
os.makedirs("output/final_csv", exist_ok=True)

df.to_csv("output/final_csv/constituency.csv", index=False, encoding="utf-8-sig")
print(f"✅ Export {len(df)} rows → output/final_csv/constituency.csv")


✅ Export 2560 rows → output/final_csv/constituency.csv
